# Ablações A/B/C — Transformers (Kaggle GPU)

Roda Grid+CV (epochs=40, patience=6) para os 6 Transformers nas ablações.

**Configurar na Célula 5:** escolher qual ablação rodar (A, B ou C).

| Ablação | Datasets | N | Estimativa T4 |
|---------|----------|---|---------------|
| B (5D ruído) | TWS_5f TWM_5f TWC_5f | 400 | ~4h |
| C (MK5) | MKE MKM MKH | 400 | ~4h |
| A (N=2k) | TWS_2k TWM_2k TWC_2k | 2000 | ~20h (resume necessário) |

**Resume:** baixa o JSON, sobe como Kaggle dataset, ajusta `RESUME_PATH` na Célula 6.

In [ ]:
# ── Célula 1: Verifica GPU ──────────────────────────────────────────────────
import torch
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'Memória: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    raise RuntimeError('GPU não detectada — ative em Settings > Accelerator')

In [ ]:
# ── Célula 2: Clonar repo ───────────────────────────────────────────────────
import os, subprocess

GIT_URL    = 'https://github.com/PauloBernardo/dissertacao-estudo-comparativo.git'
PROJECT_DIR = '/kaggle/working/sparse-lssvm-transformers-study'

if os.path.exists(PROJECT_DIR):
    subprocess.run(['git', '-C', PROJECT_DIR, 'pull', '--rebase'], check=True)
else:
    subprocess.run(['git', 'clone', GIT_URL, PROJECT_DIR], check=True)

os.chdir(PROJECT_DIR)
!git log --oneline -3
print(f'Dir: {os.getcwd()}')

In [ ]:
# ── Célula 3: Dependências ──────────────────────────────────────────────────
!pip install -q entmax einops scikit-posthocs
import torch, sklearn, numpy
print(f'torch {torch.__version__} | sklearn {sklearn.__version__} | numpy {numpy.__version__}')

In [ ]:
# ── Célula 4: Datasets Tier 1 (sintéticos gerados automaticamente) ──────────
!python scripts/download_data.py --tier 1
!ls -lh data/raw/

In [ ]:
# ── Célula 5: CONFIGURAÇÃO — escolha a ablação ──────────────────────────────
# Descomente UMA das opções abaixo:

# ABLATION = 'A'  # N=400 → N=2000 (~6h com 20 seeds)
ABLATION = 'BC'   # B + C juntos (~6h com 30 seeds, cabe numa sessão de 9h)
# ABLATION = 'B'  # 2D → 5D ruído (~3h)
# ABLATION = 'C'  # MK5 multifeature (~3h)

# Seeds: A usa 20 (LSSVM+XGB rodou com 20); B/C usam 30 (LSSVM+XGB rodou com 30)
N_SEEDS = 20 if ABLATION == 'A' else 30
SEEDS = list(range(N_SEEDS))
SEEDS_STR = ' '.join(str(s) for s in SEEDS)

ABLATION_CONFIG = {
    'A':  {'datasets': 'TWS_2k TWM_2k TWC_2k',                   'output': 'results/ablation_a_transformers.json'},
    'B':  {'datasets': 'TWS_5f TWM_5f TWC_5f',                   'output': 'results/ablation_b_transformers.json'},
    'C':  {'datasets': 'MKE MKM MKH',                             'output': 'results/ablation_c_transformers.json'},
    'BC': {'datasets': 'TWS_5f TWM_5f TWC_5f MKE MKM MKH',       'output': 'results/ablation_bc_transformers.json'},
}

cfg = ABLATION_CONFIG[ABLATION]
OUTPUT_FILE = cfg['output']
DATASETS    = cfg['datasets']
n_datasets  = len(DATASETS.split())
print(f'Ablação {ABLATION}: {n_datasets} datasets × {N_SEEDS} seeds × 6 modelos = {n_datasets*N_SEEDS*6} entries')
print(f'Datasets: {DATASETS}')
print(f'Seeds: 0..{N_SEEDS-1}')
print(f'Output: {OUTPUT_FILE}')

In [ ]:
# ── Célula 6: Resume (opcional) ─────────────────────────────────────────────
import shutil, json
from pathlib import Path

out = Path(OUTPUT_FILE)
out.parent.mkdir(exist_ok=True)

# Se quiser retomar sessão anterior:
# 1. Suba o JSON como Kaggle dataset
# 2. Ajuste o caminho abaixo e descomente

# RESUME_PATH = Path('/kaggle/input/SEU-DATASET/ablation_X_transformers.json')
# if RESUME_PATH.exists():
#     shutil.copy(RESUME_PATH, out)
#     n = len(json.load(open(out)))
#     print(f'Restaurado: {n} entries')

if out.exists():
    n = len(json.load(open(out)))
    print(f'Iniciando com {n} entries já presentes')
else:
    print('Começando do zero')

In [ ]:
# ── Célula 7: Inspecionar grade ─────────────────────────────────────────────
import sys; sys.path.insert(0, '.')
from src.tuning.grids import GRIDS, grid_size

TRANSFORMER_MODELS = [
    'FTTransformer_softmax', 'FTTransformer_topk',
    'FTTransformer_entmax', 'FTTransformer_sparsemax',
    'SAINTColnorm', 'FTTransformerCURColnorm',
]

n_datasets = len(DATASETS.split())
print(f'Ablação {ABLATION} — {n_datasets} datasets × {N_SEEDS} seeds × 6 modelos = {n_datasets*N_SEEDS*6} entries')
print(f'{"Modelo":<28}{"Grid":>6}{"Fits/entry":>12}')
print('-'*48)
total_fits = 0
for m in TRANSFORMER_MODELS:
    g = grid_size(m)
    fits = g * 5
    total_fits += fits * n_datasets * N_SEEDS
    print(f'{m:<28}{g:>6}{fits:>12}')
print(f'\nTotal fits: {total_fits:,}')

In [ ]:
# ── Célula 8: Rodar ─────────────────────────────────────────────────────────
import shutil
from pathlib import Path

models_str = ' '.join(TRANSFORMER_MODELS)

!python -u scripts/run_tier1_gridcv.py \
    --models {models_str} \
    --datasets {DATASETS} \
    --seeds {SEEDS_STR} \
    --output {OUTPUT_FILE} \
    2>&1 | tee /kaggle/working/run_ablation_{ABLATION}.log

# Copia imediatamente para o Output raiz — acessível mesmo se a sessão cair
fname = Path(OUTPUT_FILE).name
dest = Path('/kaggle/working') / fname
shutil.copy(OUTPUT_FILE, dest)
print(f'✓ Salvo em Output: {fname}  ({dest.stat().st_size/1024:.0f} KB)')

In [ ]:
# ── Célula 9: Resumo + download ─────────────────────────────────────────────
import json, statistics as st, shutil
from collections import defaultdict
from pathlib import Path

records = json.load(open(OUTPUT_FILE))
ok = [r for r in records if r.get('status') == 'ok']
n_datasets = len(DATASETS.split())
total = n_datasets * N_SEEDS * 6
print(f'Ablação {ABLATION}: {len(ok)}/{total} ({len(ok)/total*100:.0f}%)')

# Se BC, mostra B e C separados
if ABLATION == 'BC':
    B_ds = {'TWS_5f', 'TWM_5f', 'TWC_5f'}
    C_ds = {'MKE', 'MKM', 'MKH'}
    for label, dset_filter in [('B (ruído 5D)', B_ds), ('C (MK5)', C_ds)]:
        subset = [r for r in ok if r['dataset'] in dset_filter]
        f1 = defaultdict(list)
        for r in subset:
            f1[r['variant']].append(r['test_f1_macro'])
        print(f'\n── Ablação {label} ──')
        print(f'{"Modelo":<30}  F1-macro  n')
        for v, vals in sorted(f1.items(), key=lambda x: -st.mean(x[1])):
            print(f'{v:<30}  {st.mean(vals):.4f}  {len(vals)}')
else:
    f1 = defaultdict(list)
    for r in ok:
        f1[r['variant']].append(r['test_f1_macro'])
    print(f'\n{"Modelo":<30}  F1-macro  n')
    for v, vals in sorted(f1.items(), key=lambda x: -st.mean(x[1])):
        print(f'{v:<30}  {st.mean(vals):.4f}  {len(vals)}')

# Copiar para Output do Kaggle
fname = Path(OUTPUT_FILE).name
shutil.copy(OUTPUT_FILE, f'/kaggle/working/{fname}')
print(f'\nArquivo disponível em Output: {fname}')